In [1]:
import torch
from types import SimpleNamespace
from models.meshtok import MeshTok

In [2]:
def _ns(d):
    if isinstance(d, dict):
        obj = SimpleNamespace(**{k: _ns(v) for k, v in d.items()})
        obj.get = lambda k, default=None, _obj=obj: getattr(_obj, k, default)
        return obj
    if isinstance(d, list):
        return [_ns(x) for x in d]
    return d

In [3]:

device = "cuda" if torch.cuda.is_available() else "cpu"
bs = 2
x_num = 128         
data_dim = 4         
input_len = 10      
gen_len = 10        
t_total = input_len + gen_len  # 20



cfg = _ns({
    "name": "meshtok_auto",

    "all_exp": 16,
    "training": True,
    "topk": 4,
    "n_shared_experts": 2,
    "n_layer": 8,
    "dim_emb": 512,
    "moe_intermediate_size": 256,
    "dim_ffn": 1280,
    "dropout": 0.0,
    "attn_dropout": 0.0,
    "n_head": 8,
    "norm_first": True,
    "positional_embedding": None,
    "qk_norm": True,       # YAML: 1
    "norm": "rms",
    "activation": "swiglu",
    "rotary": False,       # YAML: 0

    "refine_ratio": 0.25,

    "flex_attn": False,    # YAML: 0
    "kv_cache": True,      # YAML: 1


    "dense": [True] * 8,

    "patch_num": 8,
    "patch_num_output": 8,

    "embedder": {
        "type": "conv",
        "dim": 512,
        "patch_num": 8,
        "patch_num_output": 8,
        "time_embed": "learnable",
        "select": "physical",
        "max_time_len": 20,

        "conv_dim": 32,
        "early_conv": False,  # YAML: 0
        "deep": False,        # YAML: 0
    },
})
cfg.get = lambda k, default=None: getattr(cfg, k, default)



model = MeshTok(cfg, x_num=x_num, max_output_dim=data_dim, max_data_len=t_total).to(device)
model.eval()


MeshTok(
  (embedder): ConvEmbedder(
    (positional_encoding_3d): SinusoidalPositionalEncoding3D()
    (in_proj): Conv2d(4, 512, kernel_size=(16, 16), stride=(16, 16), bias=False)
    (conv_proj): Sequential(
      (0): GELU(approximate='none')
      (1): Conv2d(512, 512, kernel_size=(1, 1), stride=(1, 1), bias=False)
    )
    (in_proj_sub): Conv2d(4, 512, kernel_size=(8, 8), stride=(8, 8), bias=False)
    (conv_proj_sub): Sequential(
      (0): GELU(approximate='none')
      (1): Conv2d(512, 512, kernel_size=(1, 1), stride=(1, 1), bias=False)
    )
    (post_proj): Sequential(
      (0): Rearrange('b (h w) d -> b d h w', h=8, w=8)
      (1): ConvTranspose2d(512, 32, kernel_size=(16, 16), stride=(16, 16), bias=False)
    )
    (post_proj_sub): Sequential(
      (0): Rearrange('b (h w) d -> b d h w', h=16, w=16)
      (1): ConvTranspose2d(512, 32, kernel_size=(8, 8), stride=(8, 8), bias=False)
    )
    (head): Sequential(
      (0): GELU(approximate='none')
      (1): Conv2d(32, 32, 

In [4]:
times = torch.linspace(0, 1, t_total, device=device)[None, :, None].repeat(bs, 1, 1)    # (bs, 20, 1)
data_input = torch.randn(bs, input_len, x_num, x_num, data_dim, device=device)         # (bs, 10, 128,128,4)
data_mask = torch.ones(bs, 1, x_num, x_num, data_dim, device=device)                   # (bs, 1, 128,128,4)

with torch.no_grad():
    out = model(
        "generate",
        data_input=data_input,
        times=times,
        input_len=input_len,
        data_mask=data_mask,
        carry_over_c=-1,
    )

print("generate out:", out.shape) 

generate out: torch.Size([2, 10, 128, 128, 4])
